# LeetCode 3721 — Longest Balanced Subarray II bằng Python

- Link: https://leetcode.com/problems/longest-balanced-subarray-ii/
- Độ khó: **Hard**
- Chủ đề: Đếm phần tử **phân biệt** theo đầu phải cố định, **lần xuất hiện trước** (`last`), Segment Tree **cộng đoạn (lazy)** + min/max, tìm **số 0 trái nhất** nhờ tính "liên tục bước ±1"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestBalanced(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Cố định đầu phải `r`, định nghĩa `D_r[l]` = (số **lẻ phân biệt**) − (số **chẵn phân biệt**) trong `nums[l..r]`. Mảng con cân bằng ⇔ `D_r[l] = 0`.
2. Khi thêm `x = nums[r]` có lần xuất hiện trước `p`: chỉ các `l ∈ (p, r]` thấy `x` là phần tử **mới** ⇒ cộng đoạn `±1`.
3. Tính chất then chốt: `|D_r[l] − D_r[l+1]| ≤ 1` ⇒ trong một đoạn, nếu `min ≤ 0 ≤ max` thì **chắc chắn có số 0** (định lý giá trị trung gian rời rạc).
4. Cài **Segment Tree** cộng đoạn + min/max, đi xuống tìm `l` trái nhất có `D = 0` trong `O(log n)`.
5. Viết bản **không đệ quy** đủ nhanh cho Python.

## 2. Đề bài & ví dụ

Mảng con **cân bằng** nếu số **giá trị chẵn phân biệt** bằng số **giá trị lẻ phân biệt** trong nó. Trả về độ dài mảng con cân bằng dài nhất (0 nếu không có).

**Ví dụ 1:** `[2,5,4,3]` → `4`.

**Ví dụ 2:** `[3,2,2,5,4]` → `5`.

**Ví dụ 3:** `[1,2,3,2]` → `3` (`[2,3,2]`).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`.

In [1]:
from typing import List
import random, time, sys

examples = [([2, 5, 4, 3], 4), ([3, 2, 2, 5, 4], 5), ([1, 2, 3, 2], 3)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[2, 5, 4, 3] -> kỳ vọng 4
[3, 2, 2, 5, 4] -> kỳ vọng 5
[1, 2, 3, 2] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Đếm phân biệt theo đầu phải

Với đầu phải `r` cố định, giá trị `x` được tính (là "phân biệt") trong `nums[l..r]` ⇔ `l ≤ last_r(x)` (lần xuất hiện cuối của `x` tới `r`). Khi `r` tăng và `nums[r] = x` từng xuất hiện ở `p`, đóng góp của `x` dịch từ các `l ≤ p` sang các `l ≤ r` ⇒ chỉ các `l ∈ (p, r]` **nhận thêm** `x`.

In [2]:
def D_table(nums):
    n = len(nums)
    for r in range(n):
        row = []
        for l in range(r + 1):
            seg = nums[l:r + 1]
            row.append(len({v for v in seg if v % 2}) - len({v for v in seg if v % 2 == 0}))
        print(f"r={r} (x={nums[r]}): D_r[0..r] = {row}")

D_table([1, 2, 3, 2])

r=0 (x=1): D_r[0..r] = [1]
r=1 (x=2): D_r[0..r] = [0, -1]
r=2 (x=3): D_r[0..r] = [1, 0, 1]
r=3 (x=2): D_r[0..r] = [1, 0, 0, -1]


### 3.2. Bước nhảy tối đa 1

`nums[l..r]` và `nums[l+1..r]` khác nhau đúng một phần tử `nums[l]`: bỏ nó đi làm số phân biệt của **một** loại giảm tối đa 1 ⇒ `|D_r[l] − D_r[l+1]| ≤ 1`. Vì vậy dãy `D_r[0..r]` "liên tục": đi từ giá trị âm sang dương phải đi qua 0.

### 3.3. Segment Tree cộng đoạn + min/max (không đệ quy)

Mỗi nút lưu `mn`, `mx` của đoạn con **đã gồm** phần cộng `ad` của chính nút đó; giá trị thật = `mn[node] + Σ ad(tổ tiên)`. Cộng đoạn bottom-up: áp `ad` lên các nút phủ, rồi cập nhật lại tổ tiên của hai biên.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Duyệt `r = 0..n−1`:

1. `s = +1` nếu `nums[r]` lẻ, `−1` nếu chẵn; `p = last[nums[r]]` (hoặc `−1`).
2. Cộng `s` vào `D[l]` với mọi `l ∈ [p+1, r]`.
3. Tìm `l` **nhỏ nhất** trong `[0, r]` có `D[l] = 0` ⇒ ứng viên `r − l + 1`.

Ví dụ 3 `[1,2,3,2]`: trước bước `r = 3`, `D_2 = [1, 0, 1]` (đoạn `[1,2,3]`: lẻ `{1,3}`, chẵn `{2}`). Tại `r = 3` (x = 2, đã xuất hiện ở `p = 1`), chỉ cộng `−1` vào `l ∈ [2, 3]` ⇒ `D_3 = [1, 0, 0, −1]`. `l` trái nhất có `D = 0` là `1` ⇒ mảng con `[2,3,2]` dài 3. ✔ Lưu ý `l = 0` **không** được cộng vì đoạn `[1,2,3,2]` đã chứa số 2 từ trước.

## 5. Cách 1 — Brute Force: hai vòng lặp + tập

In [3]:
def brute(nums):
    n = len(nums); best = 0
    for i in range(n):
        ev, od = set(), set()
        for j in range(i, n):
            (od if nums[j] % 2 else ev).add(nums[j])
            if len(ev) == len(od):
                best = max(best, j - i + 1)
    return best

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

4 kỳ vọng 4
5 kỳ vọng 5
3 kỳ vọng 3


**Nút thắt:** `O(n²) = 10^10` với `n = 10^5` (đây là lời giải của **bản I** với `n ≤ 1500`).

## 6. Ý tưởng thuật toán / bước đột phá

Hai thao tác trên mảng `D[0..n−1]` (khởi tạo 0):

- **cộng đoạn** `[p+1, r]` thêm `s`;
- **tìm `l` nhỏ nhất trong `[0, r]` với `D[l] = 0`**.

Nhờ 3.2, trong mọi đoạn con `[a, b] ⊆ [0, r]`: có số 0 ⇔ `min ≤ 0 ≤ max`. Vậy đi xuống cây: tại nút nằm trọn trong `[0, r]`, nếu con trái thoả `min ≤ 0 ≤ max` thì vào con trái, ngược lại vào con phải. Mỗi bước `O(log n)` ⇒ tổng `O(n log n)`.

## 7. Các cách tiếp theo

- **Cách 2:** mô phỏng mảng `D` trực tiếp (cộng đoạn và quét tìm 0 bằng vòng lặp) — `O(n²)` nhưng thể hiện rõ phép biến đổi.
- **Cách 3:** Segment Tree lazy **đệ quy** — `O(n log n)`, dễ đọc.
- **Cách 4:** Segment Tree **không đệ quy** (bottom-up cộng đoạn, ngăn xếp khi tìm) — `O(n log n)`, nhanh trong Python (lời giải nộp).

### 7.1. Cách 2 — Mảng `D` trực tiếp

In [4]:
def direct_D(nums):
    n = len(nums); D = [0] * n; last = {}; best = 0
    for r, x in enumerate(nums):
        s = 1 if x % 2 else -1
        for l in range(last.get(x, -1) + 1, r + 1):
            D[l] += s
        last[x] = r
        for l in range(r + 1):
            if D[l] == 0:
                best = max(best, r - l + 1); break
    return best

for a, exp in examples:
    print(direct_D(a), "kỳ vọng", exp)

4 kỳ vọng 4
5 kỳ vọng 5
3 kỳ vọng 3


### 7.2. Cách 3 — Segment Tree lazy đệ quy

In [5]:
def seg_recursive(nums):
    n = len(nums)
    mn = [0] * (4 * n); mx = [0] * (4 * n); lz = [0] * (4 * n)

    def push(x):
        if lz[x]:
            for c in (2 * x, 2 * x + 1):
                mn[c] += lz[x]; mx[c] += lz[x]; lz[c] += lz[x]
            lz[x] = 0

    def update(x, lo, hi, a, b, v):
        if b < lo or hi < a: return
        if a <= lo and hi <= b:
            mn[x] += v; mx[x] += v; lz[x] += v; return
        push(x); mid = (lo + hi) // 2
        update(2 * x, lo, mid, a, b, v); update(2 * x + 1, mid + 1, hi, a, b, v)
        mn[x] = min(mn[2 * x], mn[2 * x + 1]); mx[x] = max(mx[2 * x], mx[2 * x + 1])

    def find(x, lo, hi, r):                    # l nhỏ nhất trong [lo, min(hi, r)] có D = 0
        if lo > r or mn[x] > 0 or mx[x] < 0: return -1
        if lo == hi: return lo
        push(x); mid = (lo + hi) // 2
        res = find(2 * x, lo, mid, r)
        return res if res != -1 else find(2 * x + 1, mid + 1, hi, r)

    last = {}; best = 0
    for r, x in enumerate(nums):
        update(1, 0, n - 1, last.get(x, -1) + 1, r, 1 if x % 2 else -1)
        last[x] = r
        l = find(1, 0, n - 1, r)
        if l != -1: best = max(best, r - l + 1)
    return best

for a, exp in examples:
    print(seg_recursive(a), "kỳ vọng", exp)

4 kỳ vọng 4
5 kỳ vọng 5
3 kỳ vọng 3


**Lưu ý:** ở `find`, điều kiện `mn > 0 or mx < 0` trên một nút **vượt quá** `r` có thể chứa giá trị của các `l > r` (luôn bằng 0 vì chưa được cộng) — vẫn đúng vì ta cắt `lo > r` và hàm trả về `l` đầu tiên **thực sự** bằng 0 ở lá. Độ sâu đệ quy chỉ `O(log n)`.

### 7.3. Cách 4 — Segment Tree không đệ quy (lời giải chính)

In [6]:
def seg_iterative(nums):
    n = len(nums); size = 1
    while size < n: size *= 2
    mn = [0] * (2 * size); mx = [0] * (2 * size); ad = [0] * (2 * size)

    def add(l, r, v):                                  # [l, r]
        l += size; r += size + 1; l0, r0 = l, r - 1
        while l < r:
            if l & 1: mn[l] += v; mx[l] += v; ad[l] += v; l += 1
            if r & 1: r -= 1; mn[r] += v; mx[r] += v; ad[r] += v
            l >>= 1; r >>= 1
        for i in (l0 >> 1, r0 >> 1):
            while i:
                mn[i] = min(mn[2 * i], mn[2 * i + 1]) + ad[i]
                mx[i] = max(mx[2 * i], mx[2 * i + 1]) + ad[i]
                i >>= 1

    def leftmost_zero(r):
        st = [(1, 0, size - 1, 0)]
        while st:
            node, lo, hi, acc = st.pop()
            if lo > r: continue
            if hi <= r:
                if mn[node] + acc <= 0 <= mx[node] + acc:
                    while node < size:                 # chắc chắn có 0 bên trong: đi xuống
                        acc += ad[node]; c = 2 * node; mid = (lo + hi) // 2
                        if mn[c] + acc <= 0 <= mx[c] + acc: node, hi = c, mid
                        else: node, lo = c + 1, mid + 1
                    return lo
                continue
            acc2 = acc + ad[node]; mid = (lo + hi) // 2
            st.append((2 * node + 1, mid + 1, hi, acc2)); st.append((2 * node, lo, mid, acc2))
        return -1

    last = {}; best = 0
    for r, x in enumerate(nums):
        add(last.get(x, -1) + 1, r, 1 if x % 2 else -1)
        last[x] = r
        l = leftmost_zero(r)
        if l != -1 and r - l + 1 > best: best = r - l + 1
    return best

for a, exp in examples:
    print(seg_iterative(a), "kỳ vọng", exp)

4 kỳ vọng 4
5 kỳ vọng 5
3 kỳ vọng 3


## 8. TRACE từng bước

In [7]:
def trace(nums):
    n = len(nums); D = [0] * n; last = {}; best = 0
    for r, x in enumerate(nums):
        s = 1 if x % 2 else -1; p = last.get(x, -1)
        for l in range(p + 1, r + 1): D[l] += s
        last[x] = r
        l0 = next((l for l in range(r + 1) if D[l] == 0), -1)
        if l0 != -1: best = max(best, r - l0 + 1)
        print(f"r={r}, x={x} ({'lẻ' if s > 0 else 'chẵn'}), p={p}: cộng {s:+d} vào [{p + 1},{r}] -> D[0..r]={D[:r + 1]}, "
              f"l trái nhất D=0: {l0} -> best={best}")
    return best

trace([1, 2, 3, 2])

r=0, x=1 (lẻ), p=-1: cộng +1 vào [0,0] -> D[0..r]=[1], l trái nhất D=0: -1 -> best=0
r=1, x=2 (chẵn), p=-1: cộng -1 vào [0,1] -> D[0..r]=[0, -1], l trái nhất D=0: 0 -> best=2
r=2, x=3 (lẻ), p=-1: cộng +1 vào [0,2] -> D[0..r]=[1, 0, 1], l trái nhất D=0: 1 -> best=2
r=3, x=2 (chẵn), p=1: cộng -1 vào [2,3] -> D[0..r]=[1, 0, 0, -1], l trái nhất D=0: 1 -> best=3


3

In [8]:
trace([3, 2, 2, 5, 4])

r=0, x=3 (lẻ), p=-1: cộng +1 vào [0,0] -> D[0..r]=[1], l trái nhất D=0: -1 -> best=0
r=1, x=2 (chẵn), p=-1: cộng -1 vào [0,1] -> D[0..r]=[0, -1], l trái nhất D=0: 0 -> best=2
r=2, x=2 (chẵn), p=1: cộng -1 vào [2,2] -> D[0..r]=[0, -1, -1], l trái nhất D=0: 0 -> best=3
r=3, x=5 (lẻ), p=-1: cộng +1 vào [0,3] -> D[0..r]=[1, 0, 0, 1], l trái nhất D=0: 1 -> best=3
r=4, x=4 (chẵn), p=-1: cộng -1 vào [0,4] -> D[0..r]=[0, -1, -1, 0, -1], l trái nhất D=0: 0 -> best=5


5

Đọc trace của `[1,2,3,2]`: ở `r = 3`, `D_3 = [1, 0, 0, −1]` ⇒ `l = 1` ⇒ mảng con `[2,3,2]` dài 3 — khớp ví dụ 3 và khớp bảng ở mục 3.1.

## 9. Code nộp LeetCode

In [9]:
class Solution:
    def longestBalanced(self, nums: List[int]) -> int:
        n = len(nums)
        size = 1
        while size < n:
            size *= 2
        mn = [0] * (2 * size)
        mx = [0] * (2 * size)
        ad = [0] * (2 * size)

        def add(l, r, v):
            l += size
            r += size + 1
            l0, r0 = l, r - 1
            while l < r:
                if l & 1:
                    mn[l] += v; mx[l] += v; ad[l] += v; l += 1
                if r & 1:
                    r -= 1; mn[r] += v; mx[r] += v; ad[r] += v
                l >>= 1; r >>= 1
            for i in (l0 >> 1, r0 >> 1):
                while i:
                    a, b = 2 * i, 2 * i + 1
                    mn[i] = (mn[a] if mn[a] < mn[b] else mn[b]) + ad[i]
                    mx[i] = (mx[a] if mx[a] > mx[b] else mx[b]) + ad[i]
                    i >>= 1

        def leftmost_zero(r):
            st = [(1, 0, size - 1, 0)]
            while st:
                node, lo, hi, acc = st.pop()
                if lo > r:
                    continue
                if hi <= r:
                    if mn[node] + acc <= 0 <= mx[node] + acc:
                        while node < size:
                            acc += ad[node]
                            c = 2 * node
                            mid = (lo + hi) >> 1
                            if mn[c] + acc <= 0 <= mx[c] + acc:
                                node, hi = c, mid
                            else:
                                node, lo = c + 1, mid + 1
                        return lo
                    continue
                acc2 = acc + ad[node]
                mid = (lo + hi) >> 1
                st.append((2 * node + 1, mid + 1, hi, acc2))
                st.append((2 * node, lo, mid, acc2))
            return -1

        last = {}
        best = 0
        for r, x in enumerate(nums):
            add(last.get(x, -1) + 1, r, 1 if x & 1 else -1)
            last[x] = r
            l = leftmost_zero(r)
            if l != -1 and r - l + 1 > best:
                best = r - l + 1
        return best

sol = Solution()
for a, exp in examples:
    got = sol.longestBalanced(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[2, 5, 4, 3] -> 4 ✔
[3, 2, 2, 5, 4] -> 5 ✔
[1, 2, 3, 2] -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Bất biến:** sau bước `r`, `D[l]` (với `l ≤ r`) = số lẻ phân biệt − số chẵn phân biệt trong `nums[l..r]`. Quy nạp: giá trị `x = nums[r]` là mới với `nums[l..r]` ⇔ không xuất hiện trong `nums[l..r−1]` ⇔ `l > p` (lần xuất hiện trước gần nhất). Các giá trị khác không đổi trạng thái.
2. **Liên tục:** `|D[l] − D[l+1]| ≤ 1` với `l < r` (3.2) ⇒ trên mọi đoạn chỉ số liên tiếp trong `[0, r]`, tập giá trị là một khoảng số nguyên liên tục ⇒ có 0 ⇔ `min ≤ 0 ≤ max`.
3. **Tìm trái nhất:** các nút phủ `[0, r]` được xét từ trái sang phải; nút đầu tiên thoả điều kiện chứa số 0 trái nhất; trong nút đó, con trái được ưu tiên nếu nó chứa 0 (theo 2) ⇒ lá cuối cùng là `l` nhỏ nhất.
4. Mảng con cân bằng dài nhất kết thúc tại `r` bắt đầu tại `l` nhỏ nhất có `D[l] = 0` ⇒ lấy max theo `r`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn + tập | `O(n²)` | `O(n)` |
| 2. Mảng `D` trực tiếp | `O(n²)` | `O(n)` |
| 3. Segment tree đệ quy | `O(n log n)` | `O(n)` |
| 4. Segment tree lặp | `O(n log n)` | `O(n)` |

## 12. Bộ test

In [10]:
METHODS = [brute, direct_D, seg_recursive, seg_iterative, lambda a: Solution().longestBalanced(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().longestBalanced([1]) == 0
assert Solution().longestBalanced([2, 2, 2]) == 0
assert Solution().longestBalanced([1, 2]) == 2
assert Solution().longestBalanced([1, 1, 1, 2]) == 4             # lẻ {1}, chẵn {2}
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3721)
for _ in range(2000):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 14))]
    b = brute(a)
    assert all(f(a) == b for f in METHODS[1:]), a
print("2000 test ngẫu nhiên: OK")

2000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn n = 10^5
for name, gen in [("giá trị tới 10^5", lambda r: r.randint(1, 10**5)), ("giá trị nhỏ", lambda r: r.randint(1, 10))]:
    rng = random.Random(5); a = [gen(rng) for _ in range(10**5)]
    t0 = time.time(); x = Solution().longestBalanced(a); t1 = time.time()
    print(f"{name}: đáp án {x}, thời gian {t1 - t0:.2f}s")
rng = random.Random(6); a = [rng.randint(1, 30) for _ in range(3000)]
assert Solution().longestBalanced(a) == seg_recursive(a) == brute(a)

giá trị tới 10^5: đáp án 93405, thời gian 1.69s


giá trị nhỏ: đáp án 100000, thời gian 1.04s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm **số lần xuất hiện** thay vì số giá trị **phân biệt**

In [14]:
def wrong_count_all(nums):
    n = len(nums); best = 0
    for i in range(n):
        e = o = 0
        for j in range(i, n):
            if nums[j] % 2: o += 1
            else: e += 1
            if e == o: best = max(best, j - i + 1)                  # SAI
    return best

print("sai:", wrong_count_all([1, 2, 3, 2]), "| đúng:", brute([1, 2, 3, 2]))

sai: 4 | đúng: 3


### Lỗi 2 — Cộng `±1` vào **mọi** `l ≤ r` (quên lần xuất hiện trước)

In [15]:
def wrong_no_last(nums):
    n = len(nums); D = [0] * n; best = 0
    for r, x in enumerate(nums):
        for l in range(r + 1): D[l] += 1 if x % 2 else -1           # SAI
        l0 = next((l for l in range(r + 1) if D[l] == 0), -1)
        if l0 != -1: best = max(best, r - l0 + 1)
    return best

print("sai:", wrong_no_last([1, 1, 1, 2]), "| đúng:", brute([1, 1, 1, 2]))

sai: 2 | đúng: 4


### Lỗi 3 — Tìm số 0 bằng "giá trị bằng 0 tại nút" thay vì `min ≤ 0 ≤ max`

Nếu chỉ đi xuống khi `min == 0` (tìm min bằng 0), ta bỏ lỡ các đoạn mà 0 nằm **giữa** min âm và max dương.

In [16]:
def wrong_min_only(nums):
    n = len(nums); D = [0] * n; last = {}; best = 0
    for r, x in enumerate(nums):
        for l in range(last.get(x, -1) + 1, r + 1): D[l] += 1 if x % 2 else -1
        last[x] = r
        # SAI: chỉ chấp nhận khi min của tiền tố [0..r] bằng 0, lấy vị trí đạt min
        m = min(D[:r + 1])
        if m == 0: best = max(best, r - D.index(0) + 1)
    return best

rng = random.Random(13)
while True:
    a = [rng.randint(1, 5) for _ in range(rng.randint(2, 10))]
    if wrong_min_only(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_min_only(a), "| đúng:", brute(a))

phản ví dụ: [3, 2, 2, 2, 2, 2] | sai: 0 | đúng: 6


### Lỗi 4 — Segment tree đệ quy trên lá `[0, n)` mà quên giới hạn `l ≤ r`

Các lá `l > r` có `D = 0` (chưa được cộng) ⇒ nếu không cắt, hàm tìm có thể trả về `l > r` (độ dài âm) hoặc bị "kéo" bởi các số 0 giả ở bên phải khi kiểm tra `min ≤ 0 ≤ max` của nút giao.

In [17]:
print("Ở bước r = 0 với nums = [2, ...], D = [-1, 0, 0, ...]: nếu không giới hạn, 'l = 1' > r bị chọn nhầm.")

Ở bước r = 0 với nums = [2, ...], D = [-1, 0, 0, ...]: nếu không giới hạn, 'l = 1' > r bị chọn nhầm.


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số phần tử **phân biệt** trong mảng con" ⇒ cố định đầu phải, mỗi giá trị chỉ được tính ở lần xuất hiện **cuối** ⇒ cập nhật đoạn `(p, r]`.
- Hiệu hai đại lượng thay đổi tối đa 1 khi bỏ một phần tử ⇒ **IVT rời rạc** ⇒ tìm số 0 bằng `min ≤ 0 ≤ max`.
- Python + segment tree lazy ⇒ ưu tiên bản lặp (bottom-up cộng, ngăn xếp khi tìm).

## 15. Biến thể của bài

1. **Bản I** (`n ≤ 1500`): vét cạn `O(n²)`.
2. **Đếm số mảng con cân bằng**: cần đếm số `l` có `D[l] = 0` — segment tree lưu (min, số lần đạt min) **không đủ** vì 0 không nhất thiết là min; có thể dùng chia khối + bảng tần suất.
3. **Phân biệt theo một tính chất khác** (chia hết cho 3, nguyên tố…): chỉ đổi dấu `s`.
4. **Không phân biệt** (đếm tất cả): tiền tố `±1` + bảng vị trí đầu tiên — `O(n)`.

In [18]:
# Biến thể 4: số lẻ = số chẵn (đếm cả trùng lặp) — tiền tố ±1 + vị trí đầu tiên
def longest_equal_count(nums):
    first = {0: -1}; s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x % 2 else -1
        if s in first: best = max(best, i - first[s])
        else: first[s] = i
    return best

rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    assert longest_equal_count(a) == wrong_count_all(a)          # cùng định nghĩa "đếm tất cả"
print("Biến thể 4 OK:", longest_equal_count([1, 2, 3, 2]))

Biến thể 4 OK: 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Hai tập | `O(n²)` | Bản I, kiểm chứng |
| 2. Mảng `D` | Biến đổi rõ ràng | `O(n²)` | Hiểu bài |
| 3. ST đệ quy | Lazy + IVT | `O(n log n)` | Dễ đọc |
| 4. ST lặp | Như 3, không đệ quy | `O(n log n)` | **Lời giải chuẩn (Python)** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `longestBalanced([2,4,6,1])`.

**Bài 2 (tính tay).** Trace `D` cho `[5,5,2,2,7]`.

**Bài 3 (điền TODO).** Viết `longest_balanced_mod3(nums)`: số phân biệt chia hết cho 3 bằng số phân biệt **không** chia hết cho 3.

**Bài 4 (tự viết lại).** Viết lại Cách 2 nhưng tìm số 0 trái nhất bằng **chia khối** (mỗi khối lưu min/max và lazy).

**Bài 5 (mini challenge).** Trả về **vị trí** `(l, r)` của mảng con cân bằng dài nhất (nếu hoà, lấy `l` nhỏ nhất).

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def longest_balanced_mod3(nums):
    # TODO: đổi dấu s theo x % 3 == 0
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def longest_balanced_pos(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mảng con chứa `1` và đúng một số chẵn. (2) `[6,1]`. (3) `2`.

**Bài 2.** (1) `r=0`: `[1]`. (2) `r=1` (5 lặp, `p=0`): cộng vào `[1,1]` ⇒ `[1,1]`; `r=2` (2): `[0,0,−1]` ⇒ `l=0`, độ dài 3. (3) `r=3` (2 lặp, `p=2`): `[0,0,−1,−1]` ⇒ 4; `r=4` (7): `[1,1,0,0,1]` ⇒ `l=2`, độ dài 3. Đáp án `4`.

**Bài 3.** (1) `s = −1` nếu `x % 3 == 0`, ngược lại `+1`. (2) Giữ nguyên cấu trúc. (3) Kiểm chứng vét cạn.

**Bài 4.** (1) Khối cỡ `B`. (2) Cộng đoạn: khối trọn dùng lazy, khối biên cập nhật từng phần tử rồi tính lại min/max. (3) Tìm: khối đầu tiên có `min ≤ 0 ≤ max`, rồi quét trong khối.

**Bài 5.** (1) Lưu `(l, r)` khi cập nhật `best`. (2) Chỉ cập nhật khi **lớn hơn hẳn** để giữ `r` nhỏ… (3) Với cùng độ dài, `l` nhỏ nhất tương ứng `r` nhỏ nhất.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().longestBalanced([2, 4, 6, 1]) == 2 == brute([2, 4, 6, 1])
assert Solution().longestBalanced([5, 5, 2, 2, 7]) == 4 == brute([5, 5, 2, 2, 7])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def longest_balanced_mod3_sol(nums):
    n = len(nums); D = [0] * n; last = {}; best = 0
    for r, x in enumerate(nums):
        s = -1 if x % 3 == 0 else 1
        for l in range(last.get(x, -1) + 1, r + 1): D[l] += s
        last[x] = r
        l0 = next((l for l in range(r + 1) if D[l] == 0), -1)
        if l0 != -1: best = max(best, r - l0 + 1)
    return best

def brute_mod3(nums):
    n = len(nums); best = 0
    for i in range(n):
        a, b = set(), set()
        for j in range(i, n):
            (a if nums[j] % 3 == 0 else b).add(nums[j])
            if len(a) == len(b): best = max(best, j - i + 1)
    return best

def blocks_solution(nums):
    n = len(nums); B = max(1, int(n ** 0.5)); nb = (n + B - 1) // B
    D = [0] * n; lz = [0] * nb; bmn = [0] * nb; bmx = [0] * nb
    def rebuild(b):
        seg = D[b * B:(b + 1) * B]; bmn[b] = min(seg); bmx[b] = max(seg)
    def add(l, r, v):
        while l <= r and l % B:
            D[l] += v; l += 1
        if l - 1 >= 0 and l - 1 <= r + 1: rebuild((l - 1) // B)
        while l + B - 1 <= r:
            lz[l // B] += v; l += B
        s = l
        while l <= r: D[l] += v; l += 1
        if s <= r: rebuild(s // B)
    def find(r):
        for b in range(r // B + 1):
            lo = b * B; hi = min((b + 1) * B - 1, r)
            if hi == (b + 1) * B - 1:
                if not (bmn[b] + lz[b] <= 0 <= bmx[b] + lz[b]): continue
            for l in range(lo, hi + 1):
                if D[l] + lz[b] == 0: return l
        return -1
    last = {}; best = 0
    for r, x in enumerate(nums):
        add(last.get(x, -1) + 1, r, 1 if x % 2 else -1); last[x] = r
        l = find(r)
        if l != -1: best = max(best, r - l + 1)
    return best

rng = random.Random(4)
for _ in range(800):
    a = [rng.randint(1, 7) for _ in range(rng.randint(1, 14))]
    assert longest_balanced_mod3_sol(a) == brute_mod3(a)
    assert blocks_solution(a) == brute(a), a
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5
def longest_balanced_pos_sol(nums):
    n = len(nums); D = [0] * n; last = {}; best = 0; pos = None
    for r, x in enumerate(nums):
        for l in range(last.get(x, -1) + 1, r + 1): D[l] += 1 if x % 2 else -1
        last[x] = r
        l0 = next((l for l in range(r + 1) if D[l] == 0), -1)
        if l0 != -1 and r - l0 + 1 > best: best = r - l0 + 1; pos = (l0, r)
    return pos

print(longest_balanced_pos_sol([1, 2, 3, 2]), longest_balanced_pos_sol([3, 2, 2, 5, 4]), longest_balanced_pos_sol([2, 2]))
assert longest_balanced_pos_sol([1, 2, 3, 2]) == (1, 3)

(1, 3) (0, 4) None


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for r, x in enumerate(nums):
    add(last.get(x, -1) + 1, r, +1 nếu x lẻ, -1 nếu chẵn)   # chỉ các l chưa thấy x
    last[x] = r
    l = leftmost_zero(r)      # nút trọn trong [0, r] có min <= 0 <= max -> chắc chắn có 0
    if l != -1: best = max(best, r - l + 1)
```

**Checklist:**

- [ ] Đếm **phân biệt**, cập nhật đoạn `(last[x], r]`?
- [ ] Tìm 0 bằng `min ≤ 0 ≤ max` (dựa trên bước nhảy ≤ 1)?
- [ ] Giới hạn tìm trong `[0, r]`?
- [ ] Không đệ quy sâu / đủ nhanh cho `n = 10^5`?
- [ ] Đã so với vét cạn?